# Unified Statistical Hypothesis Testing Runner

This notebook runs **all nine hypothesis tests (H01–H09)** across multiple metrics
(TTD, TTM, tool_calls, detection rate, mitigation rate) in a single pass.

**Inputs:**
- `data_dir` — Directory with category subdirectories containing JSON run files
- `gt_dir` — Ground truth directory with per-fault YAML files (SLA thresholds)

**Output:** Merged JSON with structure `{h01: {metric: result}, ..., h09: {metric: result}}`

In [1]:
import json
import os
import sys
import logging
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path(os.getcwd()).parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(name)s %(message)s",
)

# ---- Configure input paths ----
data_dir = project_root / "hypothesis_framework" / "data" / "input_v2"
gt_dir   = project_root / "hypothesis_framework" / "data" / "groundtruth" / "kubernetes"

print(f"Data dir:    {data_dir}")
print(f"GT dir:      {gt_dir}")
print(f"Data exists: {data_dir.exists()}")
print(f"GT exists:   {gt_dir.exists()}")

Data dir:    C:\Users\meemankgupta\Music\Project\infosys\certifier\hypothesis_framework\data\input_v2
GT dir:      C:\Users\meemankgupta\Music\Project\infosys\certifier\hypothesis_framework\data\groundtruth\kubernetes
Data exists: True
GT exists:   True


In [2]:
from hypothesis_framework.scripts.run_statistical_hypothesis import (
    run_all_hypothesis_tests,
)

result = run_all_hypothesis_tests(
    data_dir=data_dir,
    gt_dir=gt_dir,
    min_runs=30,
    alpha=0.05,
    n_resamples=10000,
    random_state=42,
)

# Check for errors
if "error" in result:
    print("ERROR:", result["message"])
    if "validation" in result:
        for cat, info in result["validation"]["per_category"].items():
            print(f"  {cat}: {info}")
else:
    print("All hypothesis tests completed successfully!")
    print(f"Total runs: {result['metadata']['total_runs']}")
    print(f"Detected:   {result['metadata']['detected_runs']}")

INFO hypothesis_framework.scripts.run_statistical_hypothesis Loading runs from C:\Users\meemankgupta\Music\Project\infosys\certifier\hypothesis_framework\data\input_v2


INFO hypothesis_framework.scripts.run_statistical_hypothesis Validating minimum run criteria (min=30)


INFO hypothesis_framework.scripts.run_statistical_hypothesis Processing continuous metric: time_to_detect


INFO hypothesis_framework.scripts.run_statistical_hypothesis Processing continuous metric: time_to_mitigate


INFO hypothesis_framework.scripts.run_statistical_hypothesis Processing continuous metric: tool_calls


INFO hypothesis_framework.scripts.run_statistical_hypothesis Processing rate metric: fault_detection_success_rate


INFO hypothesis_framework.scripts.run_statistical_hypothesis Processing rate metric: fault_mitigation_success_rate


INFO hypothesis_framework.scripts.run_statistical_hypothesis All hypothesis tests complete.


All hypothesis tests completed successfully!
Total runs: 300
Detected:   239


## Metadata & Validation

In [3]:
if "error" not in result:
    meta = result["metadata"]
    print("Timestamp:", meta["timestamp"])
    print()
    print("Per-category run counts:")
    print(f"  {'Category':<25s} {'Total':>6s} {'Detected':>10s}")
    print("-" * 43)
    for cat, info in meta["per_category"].items():
        print(f"  {cat:<25s} {info['total']:>6d} {info['detected']:>10d}")
    
    print("\nSLA Thresholds loaded:")
    for metric, sla in meta.get("sla_thresholds", {}).items():
        print(f"  {metric}:")
        for fault, val in sorted(sla.items()):
            print(f"    {fault:<28s} {val:>6.0f}")

Timestamp: 2026-04-20T10:36:29.674285+00:00

Per-category run counts:
  Category                   Total   Detected
-------------------------------------------
  application_fault             60         52
  network_fault                120         88
  resource_fault               120         99

SLA Thresholds loaded:
  time_to_detect:
    disk-fill                        60
    node-restart                     30
    pod-autoscaler                  120
    pod-cpu-hog                     120
    pod-delete                       60
    pod-dns-error                    60
    pod-memory-hog                   90
    pod-network-corruption          240
    pod-network-loss                180
    pod-network-rate-limit          180
  time_to_mitigate:
    disk-fill                       180
    node-restart                    300
    pod-autoscaler                  180
    pod-cpu-hog                     300
    pod-delete                      120
    pod-dns-error                   180


## Results Summary

In [4]:
HYPOTHESIS_NAMES = {
    "h01": "Confidence Intervals",
    "h02": "Success Rate Estimation",
    "h03": "Cross-Category Comparison",
    "h04": "Success Rate Uniformity",
    "h05": "Consistency & Predictability",
    "h06": "SLA Threshold Compliance",
    "h07": "SLA Breach Rate",
    "h08": "Tail Risk Analysis",
    "h09": "Temporal Stability",
}

if "error" not in result:
    print(f"{'Hypothesis':<10s} {'Description':<32s} {'Overall Assessment'}")
    print("=" * 100)
    for hkey in sorted(result["results"].keys()):
        desc = HYPOTHESIS_NAMES.get(hkey, hkey)
        metrics_results = result["results"][hkey]
        metric_parts = []
        for mk, mr in metrics_results.items():
            if isinstance(mr, dict):
                st = mr.get("status", "")
                if st == "error":
                    metric_parts.append(f"{mk}=ERROR")
                elif st == "skipped":
                    metric_parts.append(f"{mk}=SKIPPED")
                else:
                    oa = mr.get("overall_assessment", "OK")
                    metric_parts.append(f"{mk}={oa}")
            else:
                metric_parts.append(f"{mk}=OK")
        joined = ", ".join(metric_parts)
        print(f"{hkey:<10s} {desc:<32s} {joined}")

Hypothesis Description                      Overall Assessment
h01        Confidence Intervals             time_to_detect=precise, time_to_mitigate=precise, tool_calls=precise
h02        Success Rate Estimation          fault_detection_success_rate=worst_certified_floor=64.8%, fault_mitigation_success_rate=worst_certified_floor=40.4%
h03        Cross-Category Comparison        time_to_detect=significant_category_disparity, time_to_mitigate=significant_category_disparity, tool_calls=significant_category_disparity
h04        Success Rate Uniformity          fault_detection_success_rate=uniform_rates, fault_mitigation_success_rate=non_uniform_rates
h05        Consistency & Predictability     time_to_detect=variance_instability_detected, time_to_mitigate=variance_instability_detected, tool_calls=consistent
h06        SLA Threshold Compliance         time_to_detect=sla_non_compliant, time_to_mitigate=sla_non_compliant, tool_calls=sla_compliant
h07        SLA Breach Rate                  tim

## Detailed Results Per Hypothesis

Each hypothesis uses different fields for category-level results:
- **H01**: IQM + Bootstrap CI
- **H02**: Success rate + Wilson CI + certified floor
- **H03**: Kruskal-Wallis / ANOVA heterogeneity
- **H04**: Chi-square / Fisher uniformity
- **H05**: CV stability flags
- **H06**: SLA compliance verdict (PASS/FAIL/CONDITIONAL)
- **H07**: SLA breach rate verdict
- **H08**: Tail risk level (mild/moderate/significant)
- **H09**: Drift verdict (STABLE/DRIFT_DETECTED/LOW_POWER)

In [5]:
def get_category_summary(hkey, cat_data):
    """Extract the most informative per-category summary for each hypothesis."""
    name = cat_data.get("category", "?")
    n = cat_data.get("n", cat_data.get("trials", "?"))
    
    if hkey == "h01":
        iqm = cat_data.get("iqm", 0)
        ci_lo = cat_data.get("ci_lower", 0)
        ci_hi = cat_data.get("ci_upper", 0)
        return f"{name:<25s} n={n:>3}  IQM={iqm:>8.2f}  CI=[{ci_lo:.2f}, {ci_hi:.2f}]"
    
    elif hkey == "h02":
        rate = cat_data.get("rate", 0)
        floor = cat_data.get("certified_floor", 0)
        return f"{name:<25s} n={n:>3}  rate={rate*100:>5.1f}%  certified_floor={floor*100:.1f}%"
    
    elif hkey == "h03":
        hetero = cat_data.get("within_heterogeneous", False)
        iqm = cat_data.get("equal_weight_iqm", cat_data.get("pooled_iqm", 0))
        tag = "HETEROGENEOUS" if hetero else "homogeneous"
        return f"{name:<25s} n={n:>3}  IQM={iqm:>8.2f}  within={tag}"
    
    elif hkey == "h04":
        hetero = cat_data.get("within_heterogeneous", False)
        rate = cat_data.get("rate", 0)
        tag = "NON-UNIFORM" if hetero else "uniform"
        return f"{name:<25s} n={n:>3}  rate={rate*100:>5.1f}%  within={tag}"
    
    elif hkey == "h05":
        cv = cat_data.get("pooled_cv", 0)
        flag = cat_data.get("cv_flag", "?")
        return f"{name:<25s} n={n:>3}  CV={cv:.3f}  flag={flag}"
    
    elif hkey in ("h06", "h07"):
        verdict = cat_data.get("verdict", "?")
        icon = {"PASS": "\u2705", "FAIL": "\u274c", "CONDITIONAL": "\u26a0\ufe0f",
                "INCOMPLETE": "\u2753", "INCONCLUSIVE": "\u2753"}.get(verdict, "\u2022")
        return f"{name:<25s} n={n:>3}  {icon} {verdict}"
    
    elif hkey == "h08":
        level = cat_data.get("risk_level", "?")
        icon = {"mild": "\u2705", "moderate": "\u26a0\ufe0f",
                "significant": "\u274c"}.get(level, "\u2022")
        return f"{name:<25s} n={n:>3}  {icon} risk_level={level}"
    
    elif hkey == "h09":
        dv = cat_data.get("drift_verdict", "?")
        icon = {"STABLE": "\u2705", "DRIFT_DETECTED": "\u274c",
                "LOW_POWER": "\u26a0\ufe0f"}.get(dv, "\u2022")
        return f"{name:<25s} n={n:>3}  {icon} {dv}"
    
    return f"{name:<25s} n={n:>3}"

In [6]:
if "error" not in result:
    for hkey in sorted(result["results"].keys()):
        desc = HYPOTHESIS_NAMES.get(hkey, hkey)
        print(f"\n{"=" * 80}")
        print(f"{hkey.upper()}: {desc}")
        print("=" * 80)
        for mk, mr in result["results"][hkey].items():
            print(f"\n  Metric: {mk}")
            print(f"  {"-" * 70}")
            if isinstance(mr, dict):
                st = mr.get("status", "")
                if st in ("error", "skipped"):
                    print(f"  Status: {st}")
                    if "reason" in mr:
                        print(f"  Reason: {mr['reason']}")
                    if "error" in mr:
                        print(f"  Error:  {mr['error']}")
                    continue
                oa = mr.get("overall_assessment", "N/A")
                print(f"  Overall assessment: {oa}")
                if "per_category" in mr:
                    for cat_data in mr["per_category"]:
                        summary = get_category_summary(hkey, cat_data)
                        print(f"    {summary}")


H01: Confidence Intervals

  Metric: time_to_detect
  ----------------------------------------------------------------------
  Overall assessment: precise
    application_fault         n= 52  IQM=   42.77  CI=[34.92, 49.35]
    network_fault             n= 88  IQM=  139.50  CI=[125.91, 152.33]
    resource_fault            n= 99  IQM=  109.58  CI=[100.25, 118.78]

  Metric: time_to_mitigate
  ----------------------------------------------------------------------
  Overall assessment: precise
    application_fault         n= 40  IQM=  195.14  CI=[174.62, 216.81]
    network_fault             n= 59  IQM=  285.16  CI=[264.70, 305.30]
    resource_fault            n= 73  IQM=  242.71  CI=[228.56, 258.16]

  Metric: tool_calls
  ----------------------------------------------------------------------
  Overall assessment: precise
    application_fault         n= 60  IQM=   15.00  CI=[14.06, 15.91]
    network_fault             n=120  IQM=   17.12  CI=[16.44, 17.77]
    resource_fault        

## Sub-fault Detail (H06–H09)

In [7]:
if "error" not in result:
    for hkey in ["h06", "h07", "h08", "h09"]:
        desc = HYPOTHESIS_NAMES.get(hkey, hkey)
        print(f"\n{"=" * 90}")
        print(f"{hkey.upper()}: {desc} — Sub-fault Breakdown")
        print("=" * 90)
        for mk, mr in result["results"][hkey].items():
            if not isinstance(mr, dict) or mr.get("status") in ("error", "skipped"):
                continue
            print(f"\n  Metric: {mk}")
            for cat_data in mr.get("per_category", []):
                cname = cat_data.get("category", "?")
                print(f"\n    {cname}:")
                for sf in cat_data.get("sub_faults", []):
                    sfname = sf.get("fault_name", "?")
                    sfn = sf.get("n", "?")
                    if hkey == "h06":
                        v = sf.get("verdict", "?")
                        med = sf.get("median", 0)
                        sla_val = sf.get("sla_threshold")
                        sla_str = f"SLA={sla_val:.0f}" if sla_val is not None else "no SLA"
                        wp = sf.get("wilcoxon_p")
                        wp_str = f"{wp:.4f}" if wp is not None else "\u2014"
                        icon = {"PASS": "\u2705", "FAIL": "\u274c", "CONDITIONAL": "\u26a0\ufe0f",
                                "NO_SLA_DEFINED": "\u2753"}.get(v, "\u2022")
                        print(f"      {sfname:<28s} n={sfn:>3}  median={med:>7.1f}  {sla_str:<12s}  p={wp_str:<8s}  {icon} {v}")
                    elif hkey == "h07":
                        v = sf.get("verdict", "?")
                        br = sf.get("breach_rate", 0)
                        sla_val = sf.get("sla_threshold")
                        sla_str = f"SLA={sla_val:.0f}" if sla_val is not None else "no SLA"
                        icon = {"PASS": "\u2705", "FAIL": "\u274c", "INCONCLUSIVE": "\u2753",
                                "NO_SLA_DEFINED": "\u2753"}.get(v, "\u2022")
                        print(f"      {sfname:<28s} n={sfn:>3}  breach={br*100:>5.1f}%  {sla_str:<12s}  {icon} {v}")
                    elif hkey == "h08":
                        rl = sf.get("risk_level", "?")
                        cvar = sf.get("cvar", 0)
                        iqm = sf.get("iqm", 0)
                        ratio = cvar / iqm if iqm > 0 else 0
                        icon = {"mild": "\u2705", "moderate": "\u26a0\ufe0f",
                                "significant": "\u274c"}.get(rl, "\u2022")
                        print(f"      {sfname:<28s} n={sfn:>3}  CVaR={cvar:>8.2f}  IQM={iqm:>8.2f}  ratio={ratio:.2f}  {icon} {rl}")
                    elif hkey == "h09":
                        dv = sf.get("drift_verdict", "?")
                        cusum = sf.get("cusum_drift", False)
                        ewma = sf.get("ewma_drift", False)
                        icon = {"STABLE": "\u2705", "DRIFT_DETECTED": "\u274c",
                                "LOW_POWER": "\u26a0\ufe0f"}.get(dv, "\u2022")
                        print(f"      {sfname:<28s} n={sfn:>3}  cusum={cusum}  ewma={ewma}  {icon} {dv}")


H06: SLA Threshold Compliance — Sub-fault Breakdown

  Metric: time_to_detect

    application_fault:
      node-restart                 n= 26  median=   38.6  SLA=30        p=0.9912    ❌ FAIL
      pod-delete                   n= 26  median=   49.2  SLA=60        p=0.0123    ✅ PASS

    network_fault:
      pod-dns-error                n= 22  median=   58.7  SLA=60        p=0.2723    ⚠️ CONDITIONAL
      pod-network-corruption       n= 24  median=  192.5  SLA=240       p=0.0036    ✅ PASS
      pod-network-loss             n= 21  median=  164.3  SLA=180       p=0.1519    ⚠️ CONDITIONAL
      pod-network-rate-limit       n= 21  median=  140.6  SLA=180       p=0.0008    ✅ PASS

    resource_fault:
      disk-fill                    n= 23  median=   94.4  SLA=60        p=0.9985    ❌ FAIL
      pod-autoscaler               n= 27  median=  113.1  SLA=120       p=0.1292    ⚠️ CONDITIONAL
      pod-cpu-hog                  n= 23  median=  103.9  SLA=120       p=0.0671    ⚠️ CONDITIONAL
     

## Save Merged Results

In [8]:
if "error" not in result:
    output_dir = project_root / "hypothesis_framework" / "data" / "output"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "hypothesis_results.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=4, default=str, ensure_ascii=False)

    print(f"Merged results saved to: {output_file}")
    file_kb = output_file.stat().st_size / 1024
    print(f"File size: {file_kb:.1f} KB")

Merged results saved to: C:\Users\meemankgupta\Music\Project\infosys\certifier\hypothesis_framework\data\output\hypothesis_results.json
File size: 189.6 KB
